# 🔬 Notebook Guide — ML for Omics Data

## Learning Objectives
- [ ] Build a complete RNA-seq tumor subtype classification pipeline
- [ ] Handle high-dimensional data (p >> n) correctly
- [ ] Implement cross-validation with multiple metrics simultaneously
- [ ] Apply dimensionality reduction (PCA, SelectKBest) correctly inside a Pipeline
- [ ] Perform unsupervised clustering (K-means, silhouette analysis)
- [ ] Implement a basic Kaplan-Meier survival curve
- [ ] Structure code as a reusable HackerRank submission function

## What Makes Omics ML Different
- **High dimensions**: 20,000+ genes but only hundreds of samples (p >> n)
- **Multi-collinearity**: correlated gene expression (need feature selection or regularization)
- **Imbalanced classes**: tumor subtypes with unequal frequency
- **Batch effects**: technical variation that must be removed
- **Biological interpretation**: not just accuracy — which genes drive the prediction?

---

## 🤖 Claude Integration — Copy These Prompts

**For High-Dimensional ML (p >> n problem):**
```
Explain the p >> n problem in genomics ML.
I have 200 samples and 20,000 gene expression features.
What problems does this cause for standard ML models?
What are the solutions: (1) feature selection, (2) regularization, (3) dimensionality reduction?
When should I use SelectKBest vs RFECV vs PCA vs Lasso regularization?
Show a correct Pipeline that doesn't leak feature selection into cross-validation.
```

**For Feature Importance in Genomics:**
```
I trained a RandomForest on RNA-seq data and want to know which genes matter.
Explain: feature_importances_ vs permutation_importance vs SHAP values.
What's the difference between Gini importance and permutation importance?
Why can feature_importances_ be misleading for correlated features?
How do I identify the top 20 most important genes and interpret them biologically?
```

**For Multi-class Classification Metrics:**
```
I'm classifying tumor subtypes (3 classes, imbalanced).
What are the right metrics for this problem?
Explain: macro-F1 vs weighted-F1 vs micro-F1.
What does a confusion matrix tell me that F1 doesn't?
How do I compute multi-class AUC (OVR vs OVO)?
Show me how to produce a complete classification report.
```

**For K-Means Clustering:**
```
Explain K-means clustering for tumor subtype discovery.
How do I choose K? (elbow method vs silhouette score)
What does silhouette score measure? What's a good score?
What are K-means limitations for genomics data?
What are alternatives: hierarchical clustering, DBSCAN, Gaussian mixture models?
When should I use clustering vs classification?
```

**For Kaplan-Meier Survival Analysis:**
```
Explain Kaplan-Meier survival analysis for genomics.
What is a survival curve? What is censored data?
How does KM estimate the survival probability at each time point?
What is the log-rank test for comparing two KM curves?
How would I stratify patients by gene expression for survival analysis?
Show me a minimal Kaplan-Meier implementation without lifelines.
```

**For Batch Effect Correction:**
```
Explain batch effects in RNA-seq data.
What causes batch effects? (different sequencing runs, technicians, dates)
What are the correction methods? (ComBat, limma removeBatchEffect, SVA)
How can I detect batch effects before correction? (PCA colored by batch)
When should I correct for batch effects? (before or after splitting train/test?)
```

---

## Omics ML Pipeline Template

```
RNA-seq count matrix (samples × genes)
        │
        ├── normalize (log2(counts+1) or TPM)
        ├── filter (remove low-expression genes)
        │
        ▼
        Train/Test split (stratified by subtype)
        │
        ▼
        Pipeline:
        ├── StandardScaler  (fit on train only!)
        ├── SelectKBest     (fit on train only!)
        └── RandomForest    (fit on train only!)
        │
        ▼
        Evaluate: macro-F1, AUC-OVR, confusion matrix
```

---

## Key Omics Metrics Reference

| Metric | When to Use | Formula |
|--------|-------------|---------|
| Macro-F1 | Imbalanced multiclass | Mean F1 per class |
| Weighted-F1 | When class sizes matter | F1 weighted by support |
| AUC-ROC (OVR) | Multi-class ranking | Mean of one-vs-rest AUCs |
| MCC | Binary, highly imbalanced | (TP×TN-FP×FN)/√(...) |
| Silhouette | Clustering quality | -1 to 1, higher is better |

---

## Interview Tip Bank
> **SelectKBest inside Pipeline**: `SelectKBest(f_classif, k=100)` inside a Pipeline is correct — it's fit only on training data, preventing leakage. `SelectKBest` outside Pipeline = leakage.

> **PCA for genomics**: Always scale before PCA (StandardScaler). Genes have wildly different expression ranges, so unscaled PCA would be dominated by high-variance genes regardless of biological relevance.

> **Survival curve tip**: In Kaplan-Meier, "censored" patients are those who were lost to follow-up or whose event hadn't occurred by study end. They contribute information up to their censoring time.

> **Multi-class AUC**: `roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')` is the standard. 'ovr' = one-vs-rest.

---

## Self-Check
1. You have 500 samples and 50,000 genes. Should you use PCA or SelectKBest? When would you use each?
2. Your 3-class classifier gets 90% accuracy. Is this good? What other metric would you compute first?
3. Explain what `cross_validate(pipe, X, y, scoring=['f1_macro','roc_auc_ovr_weighted'])` returns.
4. In K-means, what does a silhouette score of -0.2 mean?
5. Draw the Kaplan-Meier step for 5 patients with event times [2, 5, 5, 8, 12] and no censoring.


## TL;DR — Plain English

**What this notebook does:** Teaches you to apply machine learning to gene expression data for cancer subtype classification — a real clinical problem where you have far more features than samples.

**After this notebook you can:**
- Load and normalise RNA-seq gene expression matrices with thousands of genes
- Handle the p>>n problem (more features than samples) using PCA and regularisation
- Train classifiers to distinguish cancer subtypes from healthy tissue
- Apply survival analysis to predict patient outcomes from molecular profiles

**Why this matters:**
- HackerRank: High-dimensional ML questions (feature selection, regularisation, PCA) are core to the ML certification; the p>>n scenario is a classic trap question
- computational biology ML teams: Multi-omics integration and expression-based property prediction are active research areas; this notebook is your entry point to that domain

**Time:** ~3 hours | **Difficulty:** ⭐⭐⭐ | **Prerequisites:** 00/02 (ML fundamentals), 00/01 (Python basics)

## ML for Omics — Concepts for Beginners

| Term | Plain English |
|------|---------------|
| **omics** | Any large-scale molecular measurement: genomics (DNA), transcriptomics (RNA), proteomics (proteins) |
| **p >> n problem** | More features (genes) than samples — classical ML breaks down; needs regularisation |
| **high-dimensional data** | Data with thousands of features (genes) — visualisation and ML need special treatment |
| **PCA (Principal Component Analysis)** | Rotates data to find the directions of maximum variance — reduces thousands of genes to 2-3 axes |
| **explained variance** | What fraction of total variance each principal component captures |
| **regularisation (L1/L2)** | Penalty on large weights — L1 (Lasso) does feature selection, L2 (Ridge) shrinks all weights |
| **cross-validation** | Repeatedly split data into train/test folds to get reliable performance estimate |
| **random forest** | Ensemble of decision trees that vote on predictions — handles high-dimensional data well |
| **feature importance** | Score telling which genes contribute most to the model's predictions |
| **batch effect** | Technical variation between samples processed at different times or labs — must be corrected |
| **UMAP** | Non-linear dimensionality reduction for visualisation — better than PCA at showing clusters |
| **survival analysis** | Statistical method for time-to-event data (e.g. time to cancer recurrence) |

## Beginner Teaching Frame

**Who should fully work through this notebook:** students who know basic ML but need domain judgment for omics data.

**How to study it on a first pass:**
- focus on leakage prevention, class imbalance, and p >> n reasoning
- ask what can go wrong before asking how to improve accuracy
- treat interpretation as equally important as metrics

**Common traps:** feature selection outside CV, trusting accuracy on imbalanced data, and forgetting that biological datasets are often small and noisy.


## Canonical Resource Upgrade

Recommended companions:
- [scikit-learn MOOC](https://inria.github.io/scikit-learn-mooc/)
- [Modern Statistics for Modern Biology](https://www.huber.embl.de/msmb/)
- [Elements of Statistical Learning](https://hastie.su.domains/ElemStatLearn/)


## 📖 Prerequisites & Cross-References

**Before this notebook, you should be comfortable with:**
- **ML fundamentals** — sklearn pipelines, cross-validation, train/test splits, evaluation metrics. *Review: `00_python_ml_basics/02_ml_fundamentals.ipynb`*
- **RNA-seq data** — normalisation, differential expression, what gene expression matrices look like. *Review: `02_genomics_gene_analysis/02_rnaseq_analysis.ipynb`*
- **Statistics** — p-values, multiple testing correction, PCA intuition. *Review: `00_python_ml_basics/08_mathematical_foundations.ipynb`*

**Quick recap of terms used in this notebook:**
- **p >> n problem** — having far more features (genes) than samples (patients), causing overfitting without regularisation
- **LASSO (L1)** — regularisation that drives some coefficients to exactly zero, performing feature selection
- **PCA** — Principal Component Analysis: projects high-dimensional data onto directions of maximum variance
- **survival analysis** — modelling time-to-event data (e.g. patient survival) where some observations are censored
- **AUROC** — Area Under the ROC Curve: probability that the model ranks a random positive higher than a random negative


## What This Notebook Teaches (Plain English)

**Omics** = "everything at once" measurement of biology.
- **Genomics**: measuring all DNA
- **Transcriptomics / RNA-seq**: measuring which genes are active (how much each gene is "expressed")
- **Proteomics**: measuring all proteins

This notebook teaches ML on **RNA-seq data** — a table where rows = patients/samples, columns = genes, and values = how active each gene is. The goal: classify cancer subtypes from gene expression.

### The Challenge: Why Omics ML Is Hard

| Normal ML | Omics ML |
|-----------|---------|
| 100 features, 10,000 samples | 20,000 genes (features), 200 samples |
| Features are independent | Genes are correlated (co-expression networks) |
| Simple train/test split | Must avoid gene-selection leakage |
| Accuracy usually works | Classes severely imbalanced (rare subtypes) |

**p >> n problem**: You have more features (genes) than samples. This means almost any model will overfit unless you're careful.

### Common Pitfalls to Avoid

1. **Gene selection before splitting**: If you pick the "most variable genes" on the full dataset, then split — you've leaked test information. Always split first.
2. **Using accuracy**: If 95% of samples are subtype A, a model that always predicts A gets 95% accuracy but is useless. Use F1, MCC, or AUC.
3. **Ignoring batch effects**: Samples from different labs/days look systematically different even for the same biology.

### What You Need Before Starting
- ML Fundamentals (Notebook 00/02) — pipelines, cross-validation, metrics
- No genomics background required

# ML for Bioinformatics & Omics Data
**HackerRank Prep — Theme 4 | Interview-Grade**

Covers: gene expression classification, survival analysis, multi-omics integration, feature selection, model explainability.

> **Interview tip:** Always mention *why* you chose a method, not just *what* it is.

---

## 1. Complete ML Pipeline — Gene Expression Classification

**Scenario:** Classify tumor subtypes from RNA-seq data. Interviewers watch for:
- Correct train/test split before any preprocessing
- No data leakage through normalization
- Stratification for class imbalance
- Multiple metrics reported

### 📖 Reading Guide — PCA on Gene Expression Data

`X = np.random.randn(n_samples, n_features)`
→ *Simulated gene expression matrix: 100 patients × 1000 genes. Each value = expression level of that gene in that patient.*

`X_s = scaler.fit_transform(X)`
→ *Standardise: gene 1 might have values 0.001–0.01, gene 2 might have values 100–10000. Scaling puts all on equal footing.*

`pca = PCA(n_components=10)`
→ *Keep only the 10 most informative directions in the 1000-dimensional data.*

`X_pca = pca.fit_transform(X_s)`
→ *Project 1000 genes → 10 principal components. Each PC is a linear combination of genes.*

`pca.explained_variance_ratio_`
→ *How much variance (information) each PC captures. If PC1=40%, it summarises 40% of all gene expression variation.*



### 📦 Libraries Used Here

- **`sklearn.decomposition.PCA`** — Principal Component Analysis — compresses 1000s of genes into 2–3 dimensions you can plot.
- **`sklearn.preprocessing.StandardScaler`** — Normalise data: subtract mean, divide by std. Required before PCA — otherwise genes with big values dominate.
- **`sklearn.model_selection.train_test_split`** — Split data into training set (model learns from this) and test set (model never sees this during training).
- **`sklearn.ensemble.RandomForestClassifier`** — An ensemble of decision trees. Robust to noise, works well on high-dimensional data like gene expression.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Generic ML for Omics analysis
np.random.seed(42)
n_samples, n_features = 100, 1000  # typical omics dimensions

X = np.random.randn(n_samples, n_features)
y = np.random.randint(0, 3, n_samples)

scaler = StandardScaler()
X_s = scaler.fit_transform(X)
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_s)

print(f"Data shape: {X.shape} -> PCA: {X_pca.shape}")
print(f"Variance explained by top 10 PCs: {pca.explained_variance_ratio_.sum():.1%}")
print("Common omics ML workflow: filter -> normalize -> PCA -> cluster/classify")

> **Expected output:** Data shape (e.g., `(200, 5000) -> PCA: (200, 10)`), variance explained percentage, and workflow summary  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Generic ML for Omics analysis
np.random.seed(42)
n_samples, n_features = 100, 1000  # typical omics dimensions

X = np.random.randn(n_samples, n_features)
y = np.random.randint(0, 3, n_samples)

scaler = StandardScaler()
X_s = scaler.fit_transform(X)
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_s)

print(f"Data shape: {X.shape} -> PCA: {X_pca.shape}")
print(f"Variance explained by top 10 PCs: {pca.explained_variance_ratio_.sum():.1%}")
print("Common omics ML workflow: filter -> normalize -> PCA -> cluster/classify")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Generic ML for Omics analysis
np.random.seed(42)
n_samples, n_features = 100, 1000  # typical omics dimensions

X = np.random.randn(n_samples, n_features)
y = np.random.randint(0, 3, n_samples)

scaler = StandardScaler()
X_s = scaler.fit_transform(X)
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_s)

print(f"Data shape: {X.shape} -> PCA: {X_pca.shape}")
print(f"Variance explained by top 10 PCs: {pca.explained_variance_ratio_.sum():.1%}")
print("Common omics ML workflow: filter -> normalize -> PCA -> cluster/classify")

## 2. Feature Importance & Explainability

> **Interview key:** Interviewers love when you can explain *which genes drive the decision*.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Generic ML for Omics analysis
np.random.seed(42)
n_samples, n_features = 100, 1000  # typical omics dimensions

X = np.random.randn(n_samples, n_features)
y = np.random.randint(0, 3, n_samples)

scaler = StandardScaler()
X_s = scaler.fit_transform(X)
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_s)

print(f"Data shape: {X.shape} -> PCA: {X_pca.shape}")
print(f"Variance explained by top 10 PCs: {pca.explained_variance_ratio_.sum():.1%}")
print("Common omics ML workflow: filter -> normalize -> PCA -> cluster/classify")

## 3. Dimensionality Reduction — PCA + UMAP

> **Interview Q:** *Why use PCA before ML on gene expression data?*  
> **A:** High-dimensional data (p >> n) causes curse of dimensionality. PCA removes correlated features, reduces noise, improves generalization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Simulate gene expression data (n=200 samples, p=5000 genes)
np.random.seed(42)
n_samples, n_genes = 200, 5000
n_groups = 4

# Create groups with distinct expression profiles
X_list = []
labels = []
for g in range(n_groups):
    n_g = n_samples // n_groups
    X_g = np.random.randn(n_g, n_genes)
    # Add group-specific signal in first 100 genes
    X_g[:, g*25:(g+1)*25] += 3.0
    X_list.append(X_g)
    labels.extend([g] * n_g)

X = np.vstack(X_list)
labels = np.array(labels)

# PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=20)
X_pca = pca.fit_transform(X_scaled)

# Explained variance
var_explained = pca.explained_variance_ratio_
print(f"PC1 explains {var_explained[0]:.1%} of variance")
print(f"PC2 explains {var_explained[1]:.1%} of variance")
print(f"Top 10 PCs explain {var_explained[:10].sum():.1%} of variance")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['blue', 'red', 'green', 'orange']
for g in range(n_groups):
    mask = labels == g
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=colors[g], label=f'Group {g}', alpha=0.6, s=20)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('PCA of Gene Expression')
axes[0].legend()
axes[1].plot(range(1,21), np.cumsum(var_explained[:20]))
axes[1].set_xlabel('Number of PCs')
axes[1].set_ylabel('Cumulative variance explained')
axes[1].set_title('Scree Plot')
plt.tight_layout()
plt.savefig('/tmp/pca_omics.png', dpi=72)
print("PCA plot saved.")

## 4. Clustering — Unsupervised Discovery

> **Interview Q:** *What if you don't know the number of subtypes?*

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Generic ML for Omics analysis
np.random.seed(42)
n_samples, n_features = 100, 1000  # typical omics dimensions

X = np.random.randn(n_samples, n_features)
y = np.random.randint(0, 3, n_samples)

scaler = StandardScaler()
X_s = scaler.fit_transform(X)
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_s)

print(f"Data shape: {X.shape} -> PCA: {X_pca.shape}")
print(f"Variance explained by top 10 PCs: {pca.explained_variance_ratio_.sum():.1%}")
print("Common omics ML workflow: filter -> normalize -> PCA -> cluster/classify")

## 5. Survival Analysis Basics (Cox Model)

> Common in cancer genomics. Shows bioinformatics domain depth.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Generic ML for Omics analysis
np.random.seed(42)
n_samples, n_features = 100, 1000  # typical omics dimensions

X = np.random.randn(n_samples, n_features)
y = np.random.randint(0, 3, n_samples)

scaler = StandardScaler()
X_s = scaler.fit_transform(X)
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_s)

print(f"Data shape: {X.shape} -> PCA: {X_pca.shape}")
print(f"Variance explained by top 10 PCs: {pca.explained_variance_ratio_.sum():.1%}")
print("Common omics ML workflow: filter -> normalize -> PCA -> cluster/classify")

## 6. HackerRank ML Assessment Checklist

```
SCORING RUBRIC (what interviewers check):
✅ Split data before ANY preprocessing
✅ Use Pipeline to prevent leakage
✅ Stratified split for imbalanced classes
✅ Report 3+ metrics (don't just use accuracy)
✅ Handle class imbalance (class_weight, SMOTE)
✅ Feature selection on high-dim data
✅ Cross-validate, report mean ± std
✅ Interpret model predictions
✅ Comment on biological meaning
```

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Generic ML for Omics analysis
np.random.seed(42)
n_samples, n_features = 100, 1000  # typical omics dimensions

X = np.random.randn(n_samples, n_features)
y = np.random.randint(0, 3, n_samples)

scaler = StandardScaler()
X_s = scaler.fit_transform(X)
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_s)

print(f"Data shape: {X.shape} -> PCA: {X_pca.shape}")
print(f"Variance explained by top 10 PCs: {pca.explained_variance_ratio_.sum():.1%}")
print("Common omics ML workflow: filter -> normalize -> PCA -> cluster/classify")

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- The p>>n problem (high-dimensional statistics) — why OLS fails, bias-variance tradeoff at scale
- PCA derivation via SVD — covariance matrix eigenvectors, variance explained, scree plot interpretation
- Regularization paths — LASSO (L1, sparsity via KKT), Ridge (L2, shrinkage), Elastic Net (mixture)
- Multiple testing correction — Bonferroni (FWER), Benjamini-Hochberg (FDR), q-values, volcano plots
- Log₂ fold change math, TMM / DESeq2 / CPM normalization methods for RNA-seq
- [Elements of Statistical Learning (ESL)](https://hastie.su.domains/ElemStatLearn/) — Hastie/Tibshirani, free PDF, Ch 3+5+18

### 2️⃣ Must-Have Popular Resources
- 📘 **Elements of Statistical Learning** (Hastie/Tibshirani/Friedman) — free PDF, canonical ML theory reference
- 📘 **Modern Statistics for Modern Biology** (Holmes/Huber) — [free online](https://www.huber.embl.de/msmb/), R + biology focus
- 🎓 **StatQuest with Josh Starmer** — [YouTube playlist](https://www.youtube.com/c/joshstarmer) — PCA, UMAP, DESeq2 explained visually
- ⭐ **GitHub** [theislab/scanpy](https://github.com/scverse/scanpy) 2k★ — scRNA-seq analysis, UMAP, Leiden clustering
- ⭐ **GitHub** [scikit-learn](https://github.com/scikit-learn/scikit-learn) 59k★ — LASSO, ElasticNet, PCA, RandomForest
- 🤗 **HuggingFace** [ncats/TCGA-gene-expression](https://huggingface.co/datasets/ncats/TCGA-gene-expression) — 11k samples, 20k genes, 33 cancer types
- 📊 **Kaggle** [Gene Expression Cancer RNA-Seq](https://www.kaggle.com/datasets/crawford/gene-expression) — 5-class tumor classification, 20k features

### 3️⃣ Practicals — Hands-On
- 💻 **Exercise**: Gene expression cancer classification — NCI60 dataset, LASSO feature selection, cross-validated accuracy
- 💻 **Exercise**: PCA visualization on scRNA-seq — compute variance explained, plot elbow curve, color by cell type
- 💻 **Exercise**: Volcano plot — log₂FC vs -log10(p-value), label top 10 significant genes, apply BH correction
- 🔗 **GitHub** [CAFA-5 starter notebooks](https://github.com/mikufan/cafa-5-protein-function-starter)
- 📊 **Kaggle** [CAFA-5 Protein Function Prediction](https://www.kaggle.com/competitions/cafa-5-protein-function-prediction) — GO term annotation from sequence
- 🤗 **HuggingFace Space** [Gene expression explorer](https://huggingface.co/spaces/ncats/TCGA_visualization)

### 4️⃣ Real-World Problems
- 🏭 **Industry**: 10x Genomics scRNA-seq analysis — Seurat/Scanpy pipeline, cell type annotation, trajectory inference
- 🏭 **Industry**: Genentech/Roche oncology biomarker discovery — survival analysis, Cox regression on TCGA data
- 🏭 **Industry**: Drug response prediction — GDSC dataset, elastic net models, PRISM repurposing screen
- 📊 **Dataset**: [GTEx](https://www.gtexportal.org/) — tissue-specific expression across 54 human tissues, eQTL mapping
- 📊 **Dataset**: [CCLE](https://depmap.org/portal/) — Cancer Cell Line Encyclopedia, drug sensitivity + genomics
- 🔬 **Application**: GWAS hit functional annotation — LD-score regression, gene-set enrichment, Mendelian randomization

### 5️⃣ Interview Tips
- ❓ **Must know**: Why is p>>n a problem — rank-deficient covariance matrix, infinite solutions for OLS, need for regularization
- ❓ **Must know**: What LASSO does geometrically vs Ridge — L1 ball corners induce sparsity; L2 ball is smooth, no exact zeros
- ❓ **Must know**: When PCA is misleading — batch effects create PC1/PC2 structure unrelated to biology; always check with known covariates
- ⚠️ **Common mistake**: Performing PCA on the full dataset before train/test split — leaks test set variance into components
- ⚠️ **Common mistake**: Using raw counts for correlation-based clustering — normalize first (log1p + z-score per gene)
- 💡 **Pro tip**: In a bioinformatics interview, mention FDR (not Bonferroni) for genomics — Bonferroni is too conservative for correlated tests
- 💡 **Pro tip**: Elastic Net (alpha=0.5) almost always outperforms pure LASSO in genomics — correlated genes are selected together

### 6️⃣ Hot Industry Topics
- 🔥 **Trending** [scverse/scvi-tools](https://github.com/scverse/scvi-tools) 1.3k★ — deep generative models for single-cell (scVI, scArches, totalVI)
- 🔥 **Trending** [CellTypist](https://github.com/Teichlab/celltypist) — automated cell type annotation, logistic regression on PCA embeddings
- 🔥 **Trending** [Geneformer](https://huggingface.co/ctheodoris/Geneformer) — transformer pre-trained on 30M single-cell profiles, context-aware gene embeddings
- 🔥 **Trending** [scGPT](https://github.com/bowang-lab/scGPT) 1k★ — generative pre-training on single-cell data, zero-shot cell type transfer
- 🚀 **Build this**: scRNA-seq analysis mini-pipeline — load a 10x Genomics PBMC dataset from Scanpy, normalize, PCA, UMAP, Leiden cluster, annotate cell types, output HTML report
- 🚀 **Build this**: LASSO biomarker selector — TCGA BRCA expression data, LASSO path, cross-validated lambda, interpret top 20 genes with STRING enrichment

### Time Complexity — ML for Omics
| Operation | Time | Space | Notes |
|-----------|------|-------|-------|
| PCA (p genes, n samples) | O(n·p²+p³) | O(p²) | p>>n: use SVD on n×n |
| Sparse PCA | O(n·p·k·iter) | O(p·k) | k components |
| L1 regularization (LASSO) | O(n·p·iter) | O(p) | coordinate descent |
| Elastic Net | O(n·p·iter) | O(p) | — |
| Random Forest | O(T·n·p·log n) | O(T·n) | T trees |
| GSEA (m gene sets) | O(m·p·log p) | O(p) | p = genes |
| t-SNE | O(n²·d) | O(n²) | Barnes-Hut: O(n log n) |
| UMAP | O(n·k·log n) | O(n·k) | k = nearest neighbors |
| Survival analysis (Cox) | O(n·p²) | O(n·p) | p = covariates |

In [ ]:
import numpy as np
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

np.random.seed(42)
n_samples, n_features = 50, 500  # p>>n setting

# True model: only 10 features matter
true_coef = np.zeros(n_features)
true_coef[:10] = np.random.randn(10) * 3
X = np.random.randn(n_samples, n_features)
y = X @ true_coef + np.random.randn(n_samples)

scaler = StandardScaler()
X_s = scaler.fit_transform(X)

# Compare regularization methods
results = {}
for name, model in [
    ('Lasso', Lasso(alpha=0.1, max_iter=5000)),
    ('Ridge', Ridge(alpha=1.0)),
    ('ElasticNet', ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000))
]:
    cv_score = cross_val_score(model, X_s, y, cv=5, scoring='r2').mean()
    model.fit(X_s, y)
    n_nonzero = (np.abs(model.coef_) > 1e-4).sum()
    results[name] = {'cv_r2': cv_score, 'n_features': n_nonzero}
    print(f"{name:12s}: CV R2={cv_score:.3f}, non-zero features={n_nonzero}")

print(f"\nTrue model has {(true_coef != 0).sum()} non-zero features")
print("Lasso performs feature selection (sparse solution)")

# 🌍 Real World Problems — ML for Omics

---

## Problem 1 — TCGA PANCAN RNA-seq (5 Cancer Types)
**Dataset**: [UCI ML Repository](https://archive.ics.uci.edu/dataset/401/gene+expression+cancer+rna+seq) | [Kaggle version](https://www.kaggle.com/datasets/crawford/gene-expression)
**Size**: 801 samples × 20,531 genes, 5 cancer types
**Skills**: Full Pipeline, SelectKBest, RandomForest, stratified CV

```python
# Step 1: Download (run in terminal)
# kaggle datasets download -d crawford/gene-expression
# OR use UCI link above

# Step 2: Load
# import pandas as pd
# X = pd.read_csv('data.csv', index_col=0)
# y_df = pd.read_csv('labels.csv', index_col=0)
# y = LabelEncoder().fit_transform(y_df['Class'])

# Step 3 (run this now with simulated data):
import numpy as np
from sklearn.preprocessing import LabelEncoder
np.random.seed(42)
class_sizes = {'BRCA': 300, 'KIRC': 146, 'COAD': 78, 'LUAD': 141, 'PRAD': 136}
labels = np.concatenate([[k]*v for k, v in class_sizes.items()])
np.random.shuffle(labels)
X = np.random.lognormal(3, 1.5, (801, 500))  # 500 genes (simplified)
y = LabelEncoder().fit_transform(labels)

# YOUR TASK — full production pipeline:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, balanced_accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ('sc',  StandardScaler()),
    ('sel', SelectKBest(f_classif, k=100)),
    ('clf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=['f1_macro', 'balanced_accuracy'])
print(f"CV macro-F1:  {scores['test_f1_macro'].mean():.4f} ± {scores['test_f1_macro'].std():.4f}")
print(f"CV bal-acc:   {scores['test_balanced_accuracy'].mean():.4f}")

pipe.fit(X_train, y_train)
print(classification_report(y_test, pipe.predict(X_test), target_names=list(class_sizes.keys())))
```

**Expected**: macro-F1 > 0.95 on real PANCAN data with SelectKBest(k=200).

---

## Problem 2 — Single-Cell RNA-seq Clustering (PBMC 3k Dataset)
**Dataset**: [10x Genomics PBMC 3k](https://cf.10xgenomics.com/samples/cell/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz)
**Kaggle**: Search "PBMC single cell" on Kaggle
**GitHub**: [github.com/scverse/scanpy](https://github.com/scverse/scanpy)
**Skills**: PCA, K-means, silhouette, UMAP

```python
# Install: pip install scanpy anndata
# import scanpy as sc
# adata = sc.datasets.pbmc3k()  # auto-downloads 2700 cells × 32738 genes
# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)
# sc.pp.highly_variable_genes(adata, n_top_genes=2000)
# sc.pp.pca(adata, n_comps=50)
# sc.pp.neighbors(adata)
# sc.tl.umap(adata)
# sc.tl.leiden(adata)  # community detection clustering

# Manual approach (no scanpy needed — uses notebook skills):
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Simulate scRNA-seq structure: 2700 cells, 2000 highly variable genes
np.random.seed(42)
n_cells, n_genes = 2700, 2000
# Simulate 8 cell types
cell_types = np.repeat(np.arange(8), n_cells // 8)
X_sc = np.zeros((n_cells, n_genes))
for ct in range(8):
    mask = cell_types == ct
    signature = np.random.choice(n_genes, 200, replace=False)
    X_sc[np.ix_(mask, signature)] += np.random.lognormal(2, 0.5, (mask.sum(), 200))
X_sc += np.random.lognormal(0, 0.3, X_sc.shape)

# YOUR TASK:
# 1. Log-normalize: log1p(X / X.sum(axis=1, keepdims=True) * 10000)
X_norm = np.log1p(X_sc / X_sc.sum(axis=1, keepdims=True) * 1e4)
# 2. PCA to 50 components
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_norm)
print(f"Variance explained by 50 PCs: {pca.explained_variance_ratio_.sum():.2%}")
# 3. K-means with k=8, silhouette score
km = KMeans(n_clusters=8, random_state=42, n_init=10)
labels = km.fit_predict(X_pca)
sil = silhouette_score(X_pca, labels, sample_size=500)
print(f"K=8 Silhouette: {sil:.4f}")
# 4. Adjusted Rand Index vs true cell types
from sklearn.metrics import adjusted_rand_score
print(f"ARI vs true cell types: {adjusted_rand_score(cell_types, labels):.4f}")
```

---

## Problem 3 — Bulk RNA-seq Differential Expression (DESeq2-style in Python)
**Dataset**: [Kaggle: RNA-seq example data](https://www.kaggle.com/datasets/rana2hin/rna-seq-example-data)
**Skills**: Pandas, scipy stats, multiple testing correction

```python
import pandas as pd
import numpy as np
from scipy import stats

# Simulate DESeq2-style analysis
np.random.seed(42)
n_genes, n_samples = 1000, 20
conditions = ['control'] * 10 + ['treatment'] * 10

# Simulate count matrix (negative binomial-like)
counts = np.random.negative_binomial(5, 0.3, (n_genes, n_samples)).astype(float)

# Add DE signal: 100 genes upregulated in treatment
de_genes = np.random.choice(n_genes, 100, replace=False)
counts[de_genes, 10:] *= np.random.lognormal(1.5, 0.5, (100, 10))

# Log normalize (library size correction)
lib_sizes = counts.sum(axis=0)
counts_norm = np.log2(counts / lib_sizes * 1e6 + 1)  # log2 CPM

# t-test for each gene (Welch's t-test)
ctrl = counts_norm[:, :10]
treat = counts_norm[:, 10:]

t_stats, p_vals = stats.ttest_ind(treat, ctrl, axis=1, equal_var=False)
log2fc = treat.mean(axis=1) - ctrl.mean(axis=1)

# Multiple testing correction (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
_, adj_pvals, _, _ = multipletests(p_vals, method='fdr_bh')

results = pd.DataFrame({
    'gene': [f'GENE_{i}' for i in range(n_genes)],
    'log2FC': log2fc,
    'pvalue': p_vals,
    'padj': adj_pvals
}).sort_values('padj')

# Volcano plot criterion: |log2FC| > 1 AND padj < 0.05
sig = results[(results['padj'] < 0.05) & (results['log2FC'].abs() > 1)]
print(f"Significant DE genes: {len(sig)} / {n_genes}")
print(results.head(10))
```

---

## Resources
| Resource | URL | Purpose |
|----------|-----|---------|
| UCI PANCAN | archive.ics.uci.edu/dataset/401 | 5-class cancer RNA-seq |
| Kaggle: Gene Expression (Golub) | kaggle.com/datasets/crawford/gene-expression | Leukemia subtype |
| Kaggle: RNA-seq example | kaggle.com/datasets/rana2hin/rna-seq-example-data | DEG analysis |
| GitHub: scanpy | github.com/scverse/scanpy | scRNA-seq toolkit |
| 10x Genomics PBMC3k | 10xgenomics.com | Single-cell reference |
| Kaggle: TCGA data | kaggle.com/datasets | Multiple cancer datasets |

## Mastery Check

Before moving on, make sure you can:
1. explain why p >> n is dangerous
2. build a leakage-safe pipeline
3. choose a sensible metric for imbalance
4. interpret a PCA or clustering result cautiously


---
## 📊 Real Data — Single-Cell RNA-seq with Scanpy

The PBMC 3k dataset from 10x Genomics is the standard benchmark for single-cell analysis. It contains ~2,700 peripheral blood mononuclear cells from a healthy donor, representing 8 known cell types.

This is your first encounter with the `scanpy`/`anndata` ecosystem — the standard for single-cell bioinformatics.


## ⚠️ Python Syntax Note: List Comprehensions

The next cell uses a **list comprehension** — a compact way to build a list:
```python
# Normal loop:
result = []
for x in my_list:
    if condition(x):
        result.append(transform(x))

# List comprehension (same thing, one line):
result = [transform(x) for x in my_list if condition(x)]
```
Read it as: "give me `transform(x)` for every `x` in `my_list` where `condition(x)` is True."


In [ ]:
# SCANPY ECOSYSTEM — THE STANDARD FOR SINGLE-CELL ANALYSIS
# AnnData = Annotated Data Matrix:
#   X: count matrix (cells × genes)
#   obs: cell metadata (barcode, cluster, cell type)
#   var: gene metadata (gene name, highly variable flag)
#   obsm: lower-dimensional embeddings (PCA, UMAP)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Try to import scanpy; fall back to manual implementation if not installed
try:
    import scanpy as sc
    SCANPY_AVAILABLE = True
    print("scanpy available:", sc.__version__)
except ImportError:
    SCANPY_AVAILABLE = False
    print("scanpy not installed — run: pip install scanpy")
    print("Using manual implementation for demonstration")

# Simulate a PBMC-like dataset (mirrors real PBMC 3k structure)
np.random.seed(42)
n_cells = 500
n_genes = 1000

# 5 cell types with characteristic marker genes
cell_types = ['T cell', 'B cell', 'NK cell', 'Monocyte', 'Dendritic cell']
cell_type_fracs = [0.35, 0.20, 0.15, 0.20, 0.10]
cell_labels = np.random.choice(len(cell_types), n_cells, p=cell_type_fracs)

# Marker genes per cell type
markers = {
    0: list(range(0, 50)),     # T cell markers: CD3D, CD3E, CD4, CD8A...
    1: list(range(80, 130)),   # B cell: CD19, MS4A1, CD79A...
    2: list(range(160, 200)),  # NK: GNLY, NKG7, GZMB...
    3: list(range(240, 290)),  # Monocyte: CD14, LYZ, CST3...
    4: list(range(320, 360)),  # DC: FCER1A, CST3, IL3RA...
}

# Simulate count matrix
counts = np.random.negative_binomial(n=1, p=0.8, size=(n_cells, n_genes)).astype(float)
for cell_idx in range(n_cells):
    ct = cell_labels[cell_idx]
    for gene_idx in markers[ct]:
        counts[cell_idx, gene_idx] += np.random.negative_binomial(5, 0.4)

gene_names = [f'GENE_{i:04d}' for i in range(n_genes)]
cell_names = [f'CELL_{i:04d}' for i in range(n_cells)]
true_labels = [cell_types[l] for l in cell_labels]

print(f"Simulated PBMC-like dataset: {n_cells} cells × {n_genes} genes")
print(f"Cell type distribution:")
for ct in cell_types:
    n = true_labels.count(ct)
    print(f"  {ct:<20}: {n:3d} cells ({n/n_cells*100:.0f}%)")

In [ ]:
# STANDARD SCANPY PREPROCESSING PIPELINE
# (Manually implemented to run without scanpy installation)

# Step 1: Quality control
total_counts = counts.sum(axis=1)
n_genes_per_cell = (counts > 0).sum(axis=1)

# Step 2: Filter cells and genes (QC thresholds)
min_genes_per_cell = 50   # remove empty droplets
max_genes_per_cell = 800  # remove doublets
min_cells_per_gene = 5    # remove unexpressed genes

cell_mask = (n_genes_per_cell >= min_genes_per_cell) & (n_genes_per_cell <= max_genes_per_cell)
gene_mask = (counts > 0).sum(axis=0) >= min_cells_per_gene

counts_filt = counts[cell_mask][:, gene_mask]
labels_filt = [true_labels[i] for i in range(n_cells) if cell_mask[i]]
print(f"After QC filtering: {counts_filt.shape[0]} cells × {counts_filt.shape[1]} genes")

# Step 3: Normalization and log-transform
# Normalize each cell to 10,000 total counts (library-size normalization)
counts_norm = counts_filt * 10000 / counts_filt.sum(axis=1, keepdims=True)
counts_log = np.log1p(counts_norm)  # log(x+1)

# Step 4: Highly variable genes (HVG) — genes with high variance across cells
# This is the key step that reduces dimensionality meaningfully
gene_means = counts_log.mean(axis=0)
gene_vars = counts_log.var(axis=0)
gene_cv = np.where(gene_means > 0, gene_vars / gene_means, 0)  # coefficient of variation
hvg_mask = gene_cv > np.percentile(gene_cv, 80)  # top 20% most variable
print(f"Highly variable genes: {hvg_mask.sum()}")

counts_hvg = counts_log[:, hvg_mask]

# Step 5: PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pca = PCA(n_components=30)
X_pca = pca.fit_transform(counts_hvg)
print(f"PCA: top 5 components explain {pca.explained_variance_ratio_[:5]*100} % variance")

# Step 6: UMAP (or t-SNE if umap-learn not installed)
try:
    from umap import UMAP
    reducer = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    X_umap = reducer.fit_transform(X_pca[:, :15])
    embed_name = 'UMAP'
except ImportError:
    from sklearn.manifold import TSNE
    X_umap = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(X_pca[:, :15])
    embed_name = 't-SNE'
    print("umap-learn not found — using t-SNE. Install with: pip install umap-learn")

# Step 7: Leiden clustering (approximated with KMeans)
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

kmeans = KMeans(n_clusters=5, random_state=42)
cluster_labels = kmeans.fit_predict(X_pca[:, :15])

# ── Visualization ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
unique_types = list(set(labels_filt))
type_colors = dict(zip(unique_types, plt.cm.tab10.colors[:len(unique_types)]))

for ax, (labels, title) in zip(axes, [
    (labels_filt, f'{embed_name}: True Cell Types'),
    ([str(c) for c in cluster_labels], f'{embed_name}: Unsupervised Clusters')
]):
    unique_labs = sorted(set(labels))
    colors_loop = plt.cm.tab10.colors
    for i, lab in enumerate(unique_labs):
        mask = [l == lab for l in labels]
        ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
                   label=lab, s=10, alpha=0.7, color=colors_loop[i % 10])
    ax.set_xlabel(f'{embed_name} 1')
    ax.set_ylabel(f'{embed_name} 2')
    ax.set_title(title)
    ax.legend(fontsize=7, markerscale=2)

plt.tight_layout()
plt.savefig('pbmc_umap.png', dpi=100)
plt.show()

print(f"\nCELL CLUSTERING QUALITY:")
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(labels_filt, cluster_labels)
print(f"Adjusted Rand Index (clusters vs true types): {ari:.3f}")
print("ARI = 1.0 means perfect match; ARI = 0 means random")

In [ ]:
# MARKER GENE ANALYSIS — Identify what makes each cluster unique

# Compute mean expression per cluster
cluster_means = {}
for c in range(5):
    mask = cluster_labels == c
    cluster_means[c] = counts_log[mask].mean(axis=0)

# Find top marker genes per cluster (highest ratio vs all other clusters)
print("TOP MARKER GENES PER CLUSTER:")
print("(connecting clusters to known cell types)")
print()

for c in range(5):
    other_mean = np.mean([cluster_means[o] for o in range(5) if o != c], axis=0)
    log2fc = cluster_means[c] - other_mean
    top_genes_idx = np.argsort(log2fc)[-3:][::-1]

    # Match to known markers
    ct_idx = np.argmax(np.bincount([cell_labels[i] for i in range(n_cells)
                                    if cell_mask[i] and cluster_labels[i] == c]))
    ct_name = cell_types[ct_idx]
    print(f"  Cluster {c} → {ct_name}:")
    print(f"    Top upregulated gene indices: {top_genes_idx} (log2FC: {log2fc[top_genes_idx].round(2)})")

print()
print("IN REAL PBMC DATA:")
print("  T cells:         CD3D, CD3E, CD4/CD8A high")
print("  B cells:         CD19, MS4A1 (CD20), CD79A high")
print("  NK cells:        NKG7, GNLY, GZMB high")
print("  Monocytes:       LYZ, CD14, CST3 high")
print("  Dendritic cells: FCER1A, CLEC10A high")
print()
print("REAL DATA EXERCISE:")
print("  To run this on actual PBMC 3k data:")
print("  1. Download: https://cf.10xgenomics.com/samples/cell-exp/1.1.0/pbmc3k/")
print("     pbmc3k_filtered_gene_bc_matrices.tar.gz (free, ~10MB)")
print("  2. Load with scanpy: sc.read_10x_mtx('filtered_gene_bc_matrices/hg19/')")
print("  3. Follow this exact pipeline")
print("  4. Expected: 8 clusters, ARI ≈ 0.85 with known cell type annotations")

## 🐛 Debug Exercise — Omics ML Pipeline

Find and fix the **3 bugs** in the gene expression analysis code below.

**Expected correct output:**
```
PCA variance explained (train only): similar to full-data PCA
DE genes after BH correction: fewer than without correction
log2FC for truly upregulated gene: positive value
```

<details>
<summary>Show answer</summary>

**Bug 1 — PCA before train/test split:** Fitting PCA on all data leaks test-set variance.
Fix: fit PCA on `X_train` only, then `transform` both splits.

**Bug 2 — No multiple testing correction:** Running 1000 t-tests at α=0.05 yields ~50
false positives by chance. Fix: apply Benjamini-Hochberg FDR correction
(`statsmodels.stats.multitest.multipletests(pvals, method='fdr_bh')`).

**Bug 3 — log2FC direction inverted:** `log2(B/A)` is positive when B > A (condition B
higher). If condition A is the disease group and B is control, this labels
*down*-regulated genes as *up*-regulated. Fix: use `log2(disease / control)`.
</details>


## 🔗 Connection to Single-Cell RNA-seq (Module 02)

For single-cell data, the **p >> n problem** is even more extreme:
- Bulk RNA-seq: ~20,000 genes × 10–100 samples
- scRNA-seq: ~20,000 genes × 10,000–1,000,000 cells, BUT most entries are **zero** (dropout/sparsity)

The ML techniques from this notebook (PCA, random forests, feature selection) all apply to scRNA-seq, with these additions:
- **Graph-based clustering (Leiden)** — builds a k-NN graph of cells and finds communities (connects to Module 06 GNNs)
- **Trajectory inference (PAGA, Monocle)** — orders cells along a developmental trajectory using graph algorithms
- **Batch correction** — critical when combining scRNA-seq datasets from different labs

→ See `02_genomics_gene_analysis/05_single_cell_rnaseq.ipynb` for the full scRNA-seq workflow.

In [ ]:
# Quick demo: Leiden clustering on synthetic data (connects to Module 06 GNNs)
import numpy as np
from sklearn.neighbors import kneighbors_graph
from sklearn.datasets import make_blobs

# Simulate 3 cell types with 50 "genes"
X, true_labels = make_blobs(n_samples=300, n_features=50, centers=3, random_state=42)

# Build k-NN graph (same concept as protein residue graphs in Module 06)
knn_graph = kneighbors_graph(X, n_neighbors=15, mode='connectivity')
print(f"Cells: {X.shape[0]}, Genes: {X.shape[1]}")
print(f"k-NN graph: {knn_graph.shape}, edges: {knn_graph.nnz}")
print(f"True clusters: {len(set(true_labels))}")

# Note: real Leiden clustering uses `import leidenalg` + igraph
# Here we show the graph construction step — the key ML concept
# For full scRNA-seq pipeline, see 02/05_single_cell_rnaseq.ipynb
print("\n→ For Leiden clustering implementation, see 02/05_single_cell_rnaseq.ipynb")
print("→ For graph neural networks on similar graphs, see Module 06")

## 🌉 Bridge to Module 05 — Deep Learning

**The jump:** Module 04 used scikit-learn (`model.fit(X, y)`). Module 05 introduces PyTorch neural networks with explicit training loops.

**The conceptual shift:**
- Instead of `model.fit()` → you write a loop: forward pass → compute loss → backward pass → update weights
- Instead of tree splits → layers of weighted matrix multiplications
- The "learning" is gradient descent, not information gain

**Before starting Module 05:**
1. ✅ Make sure you understand matrix multiplication geometrically (Module 00/08)
2. ✅ Run `import torch; print(torch.__version__)` — if it errors, install PyTorch first
3. ✅ Do `00/06_pytorch_fundamentals.ipynb` if you haven't — it's the warmup
4. 📖 Read the TL;DR of `05/01` first — it tells you exactly what to expect

**What carries over:** Train/val/test splits, cross-validation, overfitting — same concepts, new tool.

In [ ]:
# DEBUG EXERCISE — Omics ML pipeline, find and fix 3 bugs
import numpy as np
from scipy.stats import ttest_ind
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

np.random.seed(42)
n_samples, n_genes = 100, 500

# Simulate gene expression: first 10 genes upregulated in disease
X = np.random.randn(n_samples, n_genes)
y = np.array([0]*50 + [1]*50)          # 0=control, 1=disease
X[y == 1, :10] += 2.0                  # disease has higher expression for genes 0-9

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# BUG 1: PCA fitted on full X — leaks test set into transformation
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X)           # should be fit on X_train only
X_train_pca = X_pca[y_train.shape[0]:]  # this slicing is also wrong
X_test_pca  = X_pca[:y_test.shape[0]:]
print(f"PCA variance explained: {pca.explained_variance_ratio_[:3].sum():.3f}")

# BUG 2: multiple t-tests without correction — inflated false positives
control = X[y == 0]
disease = X[y == 1]
pvals = np.array([ttest_ind(disease[:, g], control[:, g]).pvalue
                  for g in range(n_genes)])
sig_genes = np.sum(pvals < 0.05)
print(f"Significant genes (no correction): {sig_genes}  — many are false positives!")
# Fix: apply BH correction
# from statsmodels.stats.multitest import multipletests
# reject, pvals_corrected, _, _ = multipletests(pvals, method='fdr_bh')

# BUG 3: log2FC computed as log2(control / disease) — direction is inverted
# upregulated in disease should give POSITIVE log2FC
log2fc = np.log2(control.mean(0) + 1e-9) - np.log2(disease.mean(0) + 1e-9)  # inverted
print(f"log2FC gene 0 (truly upregulated in disease): {log2fc[0]:.3f}")
print("Expected: positive value (disease > control), got negative because formula is inverted")
